# Prefix Tuning

**Tags:** parameter-efficient, finetuning
**Prerequisites:** Fine-tuning
**Related Concepts:** See flowchart below
**Source:** llm/concepts/prefix-tuning.md

## TL;DR

Learn task-specific prefix tokens prepended to input, affecting all transformer layers. Only prefix trainable (~1-2% params). Better than LoRA for sequence-to-sequence tasks; worse for multi-task (prefix conflicts). Trade-off: simplicity vs flexibility.

## Core Intuition

Instead of training LoRA matrices or adapter layers, prefix tuning learns "soft prompt" tokens. Prepend ~50-100 learnable vectors to every input. Model learns what these vectors mean in latent space (like instruction embeddings). They attend to input, influencing all layers. Simple but less flexible than adapters.

## How It Works

**Standard Task (no tuning):**
```
Input text: "Summarize: [document]"
→ tokenize → [tok_1, tok_2, ...]
→ embedding layer → [e_1, e_2, ...]
→ transformer layers → output
```

**Prefix Tuning:**
```
Prefix: p_0, p_1, ..., p_k (learned vectors in latent space)
Document: [tok_1, tok_2, ...] → embeddings [e_1, e_2, ...]

Combined input to transformer:
  [p_0, p_1, ..., p_k, e_1, e_2, ..., e_T]

Only p_0...p_k are trained (updated via gradient)
Base model weights frozen
```

**Mathematical Formulation:**

Transformer with prefix:
```
Attention at layer l:
  Q_l = W_Q h_l
  K_l = W_K [p_l_prefix; h_l]     # prefix + hidden states
  V_l = W_V [p_l_prefix; h_l]
  
  Attn = softmax(Q_l @ K_l^T / √d) @ V_l

Prefix activation:
  p_0^l = h_0^(l-1) (from previous layer)
  where h_0^(l-1) includes attention to prefix tokens
```

**Parameterization:**

Prefix tokens can be:
1. **Direct:** p = learnable vectors (768-dim each)
   - Parameters: prefix_length × hidden_dim
   - Simple, but many parameters
   
2. **Reparameterized (better):** p = MLP(p_0)
   - p_0: small matrix (e.g., 10 × 768)
   - MLP: projects to full prefix (e.g., 10 × 100)
   - Reduces parameters, improves stability
   ```
   p = MLP(p_0)  where p_0 ∈ ℝ^(10 × 768)
   MLP = Linear(768 → 4096) → ReLU → Linear(4096 → 7680)
   Result: p ∈ ℝ^(100 × 768)
   
   Parameters: (10 × 768) + (768 → 4096) + (4096 → 7680)
             = 7,680 + 3.2M + 31.2M = ~34M (larger but more stable)
   ```

**Prefix Length & Optimal Size:**

```
Too short (<10 tokens):
  - capacity too limited
  - underfitting
  - can't encode task adequately
  
Short (10-50 tokens):
  - minimal parameters (~8-38K)
  - limited expressiveness
  - works for simple tasks
  
Medium (50-100 tokens):
  - ~38-76K parameters (with direct) or ~1-2M (reparameterized)
  - good balance
  - default recommendation
  
Long (100-500 tokens):
  - ~76-384K direct or ~5-10M reparameterized
  - higher capacity
  - risk of overfitting on small tasks
  - can handle complex tasks
```

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Prefix Tuning Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | Prefix | LoRA | Adapters | BitFit |
|--------|--------|------|----------|--------|
| Parameters | 1% | 1-2% | 2-5% | 0.1% |
| Trainability | Prefix only | Low-rank matrices | Bottleneck layers | Bias only |
| Multi-task | No (conflicts) | Yes (multi-LoRA) | Yes (stacked) | Yes |
| Quality | 92-96% vs FT | 96-98% vs FT | 96-98% vs FT | 85-92% vs FT |
| Flexibility | Low (affects all) | High (surgical) | High | Low |
| Initialization | Critical | Neutral | Neutral | Neutral |
| Combination | No conflicts possible | Can combine multiple | Can stack | Can combine |

**Task Suitability:**
```
Prefix Tuning works well:
  - Summarization (task is prepended instruction)
  - Translation (source + prefix → translation)
  - Question answering (question + prefix → answer)
  - Paraphrasing (text + prefix → paraphrase)

Not ideal:
  - Multiple tasks (prefix conflicts)
  - Fine-grained control (affects all layers equally)
  - Classification (instruction doesn't flow naturally)
```

### Code Implementation

```python
# Key imports for Prefix Tuning
import numpy as np
import torch
from typing import Any

# Prefix Tuning example implementation
class PrefixTuning:
    """
    Prefix Tuning implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Prefix Tuning"]
    B["Fine-tuning"] -->|prerequisite| A
    A -->|used with| D["LoRA (Low-Rank Adaptation)"]
    A -->|used with| D["Adapters"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Prefix Tuning used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Prefix Tuning?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Prefix Tuning compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Prefix Tuning?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Prefix Tuning?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/prefix-tuning.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]